# Evaluate IC50 for statin analogues

In [10]:
sys.path.append(os.path.join(RDConfig.RDContribDir,'SA_Score'))

In [13]:
os.path(RDConfig.RDContribDir)

TypeError: 'module' object is not callable

In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import deepchem as dc
import time
import transformers
import random
import mordred
from rdkit import Chem
import matplotlib.pyplot as plt
from rdkit.Chem import AllChem, Draw
from sklearn.model_selection import train_test_split
from deepchem.feat.smiles_tokenizer import SmilesTokenizer
from rdkit import DataStructs
from rdkit.Chem.Fingerprints import FingerprintMols

from rdkit.Chem import RDConfig
import sys, os
sys.path.append(os.path.join(RDConfig.RDContribDir,'SA_Score'))
import sascorer


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (C:\ProgramData\Anaconda3\envs\rdkitenv\lib\site-packages\deepchem\models\torch_models\__init__.py)
Skipped loading some Jax models, missing a dependency. No module named 'jax'


## Setup for Statin MLP
- Load a pre-trained MLP that predicts Statin IC50
- Set-up the featurizer needed for the Statin MLP

In [2]:
#featurizer=dc.feat.RDKitDescriptors()
#featname="RDKitDescriptors"
featurizer=dc.feat.MordredDescriptors()
featname="MordredDescriptors"

mlp_df = pd.read_csv("905-unique-statins.csv")
mlp_x = [""]*len(mlp_df)
i=0
for name in mlp_df["Ligand SMILES"]:
    mlp_x[i]=name.replace("[Na+].","").replace(".[Na+]","")
    i += 1
mlp_x = list(mlp_x)

mlp_mols = [Chem.MolFromSmiles(smile) for smile in mlp_x]
mlp_f = featurizer.featurize(mlp_mols)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scalername = "StandardScaler"
scaler.fit(mlp_f)
mlp_scaled = scaler.transform(mlp_f)

from sklearn.decomposition import PCA
pca = PCA(n_components=75)
pca.fit(mlp_scaled)

Statin_MLP = tf.keras.models.load_model("11May_MorPCA75_2B_FT_4x400AdamRelu_150epcs")
Statin_MLP.compile()
Statin_MLP.summary()

print("MLP model loaded and PCA prepared")

C:\ProgramData\Anaconda3\envs\rdkitenv\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)





Model: "model_14"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_16 (InputLayer)       [(None, 75)]              0         
                                                                 
 normalization_15 (Normaliz  (None, 75)                151       
 ation)                                                          
                                                                 
 skip_dense_block_25 (SkipD  (None, 400)               511600    
 enseBlock)                                                      
                                                                 
 skip_dense_block_26 (SkipD  (None, 800)               641600    
 enseBlock)                                                      
                                                                 
 dense_119 (Dense)           (None, 1)                 801       
                                                       

In [3]:
known_df = pd.read_csv("known_statins.csv")
#================================================================
#use this code to test chembl list
#known_df = pd.read_csv("Statin_Chembl_SMILES.csv")
#=================================================================
known_smiles = [""]*len(known_df["SMILES"])
for i,name in enumerate(known_df["SMILES"]):
    known_smiles[i]=name.replace("[Na+].","").replace(".[Na+]","").replace(".[Ca+2]","")
knowns = known_smiles

mol_pred_batch = []
for smile in knowns:
    mol = Chem.MolFromSmiles(smile)
    mol_pred_batch.append(mol)
    
featurized_pred_batch = featurizer.featurize(mol_pred_batch)
scaled_pred_batch = scaler.transform(featurized_pred_batch)
pca_pred_batch = pca.transform(scaled_pred_batch)

predicted_pred_batch = Statin_MLP.predict(pca_pred_batch)

err = []
for name, ic50,exp in zip(known_df["name"],predicted_pred_batch,known_df["exp"]):
  ic50 = np.exp(ic50)
  err.append(abs(ic50-exp))
  print(f"IC50 for {name} is {ic50} and MAE is {abs(ic50-exp)}")
print(f"average error for knows is {sum(err)/len(err)}")

1/1 [==============================] - 1s 596ms/step
IC50 for Cerivastatin is [8.56276] and MAE is [5.0227604]
IC50 for Simvastatin is [7.4131517] and MAE is [4.673152]
IC50 for Atorvastatin is [4.7751584] and MAE is [3.6151586]
IC50 for Rosuvastatin is [3.6671147] and MAE is [3.5071146]
average error for knows is [4.2045465]


## Define helper functions
- This block defines the sequential model and runs the optimization.

- can load previous weights

In [4]:
def strip_smiles(input_string):
    output_string = input_string.replace(" ","").replace("[CLS]","").replace("[SEP]","").replace("[PAD]","")
    output_string = output_string.replace("[Na+].","").replace(".[Na+]","")
    return output_string

def mols_from_smiles(input_smiles_list):
    valid_mols = np.empty([len(input_smiles_list)], dtype="object")
    valid_smiles = [""]*len(input_smiles_list)
    good_count = 0
    for ti, smile in enumerate(input_smiles_list):
       temp_mol = Chem.MolFromSmiles(smile)
       if temp_mol != None:
           valid_mols[good_count] = temp_mol
           valid_smiles[good_count] = smile
           good_count += 1
       else:
           print(f"SMILES {ti} was not valid!")
    print(f"Generated a total of {good_count} mol objects")
    return valid_mols[:good_count], valid_smiles[:good_count]

print("Statin GPT model defined.")

Statin GPT model defined.


## Analyze Lists of Molecules

In [49]:
statin_number = 663
model_size = 55
what_percent = .10
read_path = "Gen_results/"
write_path = "Gen_IC50_results/"
pic_path = "Gen_dock_results/Images/"
#read_path = "model_compare/"
#write_path = "model_compare/"
read_name = f"statin{statin_number}_{model_size}K_fullgen{int(what_percent*100)}p"
df = pd.read_csv(f"{read_path}{read_name}.csv")
df.head()

,Unnamed: 0,smiles,entropies
0,0,O[C@H](C[C@H](O)\C=C\c1c2CCCC(Cc3ccc(cc3)-c3cc...,0.000000
1,1,O[C@@H](CC1=CCC=CC=C1)C[C@H](O)/C=C/c1c2c(nn1-...,1.623730
2,2,C=C(CC(=O)[O-])C[C@H](O)/C=C/c1c2c(nn1-c1ccc(F...,1.605788
3,3,CCNC(CC(=O)[O-])C[C@H](O)/C=C/c1c2c(nn1-c1ccc(...,1.597745
4,4,C[CH]C(CC(=O)[O-])C[C@H](O)/C=C/c1c2c(nn1-c1cc...,1.586989


In [50]:
original_molecules = df["smiles"].to_list()

mol_original_batch,rev_gen_molecules = mols_from_smiles(original_molecules)
featurized_original_batch = featurizer.featurize(mol_original_batch)
scaled_original_batch = scaler.transform(featurized_original_batch)
pca_original_batch = pca.transform(scaled_original_batch)
predicted_original_batch = Statin_MLP.predict(pca_original_batch)
res_original = rev_gen_molecules
res2_original = np.exp(predicted_original_batch)

print_legend =[]
print_mols = []
print_smiles = []

mols_to_plot = list(map(lambda x: Chem.MolFromSmiles(x),res_original))

count = 0
average_ic50 = 0.0
average_ic50_top = 0.0
for ic50,good_mol,smiles in zip(res2_original,mols_to_plot,res_original):
    average_ic50 += ic50
    if ic50 < 1000.0:
        average_ic50_top += ic50
        print_legend.append(str(ic50))
        print_mols.append(good_mol)
        print_smiles.append(smiles)
        count += 1

if count == 0:
    leglist = list(map(lambda x: str(x),res2_original))
    print_mols = mols_to_plot

print("Original set of molecules used as seeds.")
average_ic50 = average_ic50/len(mol_original_batch)
average_ic50_top = average_ic50_top/count

print(f"Average IC50 for {count} top molecules is: {average_ic50_top}")
print(f"Average IC50 for {len(mol_original_batch)} top molecules is: {average_ic50}")

# Calculate simiarity to original molecule ===================================================

all_fp = [AllChem.GetMorganFingerprint(m,2) for m in mols_to_plot]
sim_array = []
for fp in all_fp:
    sim_array.append(DataStructs.TanimotoSimilarity(fp,all_fp[0])) 

# Calculate SAS ===============================================================================

sas_list = []
for mol in mols_to_plot:
    try:
        s = sascorer.calculateScore(mol)
        sas_list.append(s)
    except:
        print("Could not calculate SAS!")
        sas_list.append(-1.0)

# prepare data for CSV ========================================================================

molecules_for_docking = f"{write_path}{read_name}_IC50.csv"
total_ic50_list = list(map(lambda x: str(x[0]),res2_original))
out_dictionary = {"entropies": df["entropies"], "IC50": total_ic50_list, "SMILES": rev_gen_molecules, "SAS": sas_list, "SIMILARITY": sim_array}

out_df = pd.DataFrame.from_dict(out_dictionary)
old_len = len(out_df)
nn = out_df.drop_duplicates(subset=["IC50"])
new_len = len(nn)
if old_len != new_len:
    print(f"---->dropped duplicates and reduced from {old_len} to {new_len}")

mols = [Chem.MolFromSmiles(smile) for smile in nn["SMILES"]]
legends = [f"IC50: {ic50} nM" for ic50 in nn["IC50"]]
img = Draw.MolsToGridImage(mols, legends = legends, molsPerRow=3, subImgSize=(250,250))
pic = img.data
    
with open(pic_path+read_name+".png",'wb+') as outf:
    outf.write(pic)

nn.to_csv(molecules_for_docking)

print("CSV written.")
    
#Draw.MolsToGridImage(mols=print_mols, legends = print_legend, molsPerRow=5,maxMols=500)

Generated a total of 14 mol objects


C:\ProgramData\Anaconda3\envs\rdkitenv\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


1/1 [==============================] - 0s 36ms/step
Original set of molecules used as seeds.
Average IC50 for 14 top molecules is: [13.372668]
Average IC50 for 14 top molecules is: [13.372668]
CSV written.


In [14]:
new_df = pd.read_csv(molecules_for_docking)
new_df.describe()

,Unnamed: 0,entropies,IC50,SAS,SIMILARITY
count,51.000000,51.000000,51.000000,51.000000,51.000000
mean,50.411765,3.060311,202.496073,5.724657,0.591538
std,42.303748,0.620851,393.282631,0.917668,0.120739
min,0.000000,0.000000,5.808329,4.846811,0.365979
25%,15.000000,3.089724,16.994377,4.987023,0.598851
50%,37.000000,3.196168,36.819270,5.164669,0.630303
75%,77.500000,3.375274,138.616210,5.800350,0.650602
max,178.000000,3.563236,1640.862700,7.322866,1.000000


## Look at all

In [66]:
out_df.head(len(mols_to_plot))

,IC50,SMILES,SAS,SIMILARITY
0,10.066018,CC(C)c1cc(c(-c2ccccc2)n1\C=C\[C@@H](O)C[C@@H](...,3.833168,1.000000
1,87.91965,CC(=O)Oc1cc(C(C)C)n(/C=C/[C@@H](O)C[C@@H](O)CC...,4.042374,0.631068
2,100.911896,CCC(O)[CH]c1cc(C(C)C)n(/C=C/[C@@H](O)C[C@@H](O...,4.566081,0.613208
3,28.732632,CCCc1cc(-c2ccc(F)cc2)c(-c2ccccc2)n1/C=C/[C@@H]...,3.802711,0.790000
4,32.124798,CC(=O)[CH]c1cc(C(C)C)n(/C=C/[C@@H](O)C[C@@H](O...,4.320940,0.631068
5,77.08364,CC(C)c1cc(OCCCC=O)c(-c2ccccc2)n1/C=C/[C@@H](O)...,4.152892,0.590909
6,10.066018,CC(C)c1cc(-c2ccc(F)cc2)c(-c2ccccc2)n1/C=C/[C@H...,3.833168,1.000000
7,49.42695,CC(C)c1cc([CH]CCCC=O)c(-c2ccccc2)n1/C=C/[C@@H]...,4.436949,0.590909
8,35.001602,CC(=O)Cc1cc(C(C)C)n(/C=C/[C@@H](O)C[C@@H](O)CC...,4.098924,0.631068
9,80.09172,CCC(=O)c1cc(C(C)C)n(/C=C/[C@@H](O)C[C@@H](O)CC...,4.053139,0.631068


In [ ]:
# analyze statin training set

#pca_original_batch = pca.transform(mlp_scaled)
#predicted_original_batch = Statin_MLP.predict(pca_original_batch)
#==================================================================
#to analyze chembl list below / mlp list above
predicted_original_batch = Statin_MLP.predict(pca_pred_batch)
mlp_x = knowns
#==================================================================
res_original = mlp_x
res2_original = np.exp(predicted_original_batch)

print_legend =[]
print_mols = []
print_smiles = []

mols_to_plot = list(map(lambda x: Chem.MolFromSmiles(x),mlp_x))

count = 0
average_ic50 = 0.0
average_ic50_top = 0.0
for ic50,good_mol,smiles in zip(res2_original,mols_to_plot,mlp_x):
    average_ic50 += ic50
    if ic50 < 1000.0:
        average_ic50_top += ic50
        print_legend.append(str(ic50))
        print_mols.append(good_mol)
        print_smiles.append(smiles)
        count += 1

if count == 0:
    leglist = list(map(lambda x: str(x),res2_original))
    print_mols = mols_to_plot

print("Statin training set.")
average_ic50 = average_ic50/len(mlp_x)
average_ic50_top = average_ic50_top/count

print(f"Average IC50 for {count} top molecules is: {average_ic50_top}")
print(f"Average IC50 for {len(mlp_x)} full set of molecules is: {average_ic50}")

molecules_for_docking = "xfer_Learning_files/Statin_chembl_FULL_Molecules_21May.csv"
total_ic50_list = list(map(lambda x: str(x),res2_original))
out_dictionary = {"IC50": total_ic50_list, "SMILES": mlp_x}

out_df = pd.DataFrame.from_dict(out_dictionary)
out_df.to_csv(molecules_for_docking)

print("CSV written.")
    
Draw.MolsToGridImage(mols=print_mols, legends = print_legend, molsPerRow=5,maxMols=500)

In [ ]:
molecules_for_docking = "xfer_Learning_files/Statin_chembl__FULL_Docking_Molecules_20May.csv"
out_dictionary = {"IC50": print_legend, "SMILES": print_smiles}

out_df = pd.DataFrame.from_dict(out_dictionary)
out_df.to_csv(molecules_for_docking)

print("CSV written.")

In [ ]:
count = 0
average_ic50 = 0.0
for ic50,good_mol in zip(res2_original,mols_to_plot):
    if ic50 < 1e+50:
        average_ic50 += ic50
        count += 1
    else:
        print(f"{ic50} omitted from list")
print(f"Average ic50 for {count} molecules is {average_ic50/count}.")